In [1]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import numpy as np

In [2]:
from training.train_decoder import download_model_and_config, load_model
import einops

training decoder


In [3]:
from torch.utils.data import DataLoader
from generators.backbone_subseries_converter import EchoStateDataset
from models.vanila_decoder_transformer import TransformerDecoderModel

model_path, config = download_model_and_config('s2gk8uag')
model = load_model(model_path, config)
model.to('cuda')
model_benchmark = TransformerDecoderModel(config=config)
model_benchmark.to('cuda')
model_benchmark.eval()
config['dataset']['train']['num_series'] = 10_000
# config['dataset']['train']['series_length'] = 200
dataset_config = config.get('dataset', {})
train_dataset = EchoStateDataset(config=config, training=True, input_label=True)
val_dataset = EchoStateDataset(config=config, training=False, input_label=True)

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb:   1 of 1 files downloaded.  
/home/wojciech/private/magisterka/TFTS/training/train_decoder.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file

running backbone dataset
running backbone ESN
running backbone dataset
running backbone ESN


In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [8]:
train_dataset[0][0].shape

torch.Size([100, 2])

In [6]:
class Overlay(nn.Module):
    def __init__(self, model, emb_dim):
        super(Overlay, self).__init__()
        self.model = model
        self.linear = nn.Linear(emb_dim, 1)

    def forward(self, x):
        embb = self.model.forward_embb(x[:, :-1, 0:1])
        output = self.linear(embb)
        return output

In [11]:
from datasets import tqdm

overlay = Overlay(model_benchmark, emb_dim=128)  # Assuming the embedding dimension is 64
overlay.train()
overlay.to('cuda')
optim = torch.optim.Adam(overlay.linear.parameters(), lr=0.001)
for epoch in range(1):
    for x, y in tqdm(train_loader):
        x = x.float().to('cuda')
        y = y.float().to('cuda')
        # Forward pass through the model
        output = overlay(x)
        print(f'Output shape: {output.shape}, Target shape: {x[:, 1:, 1].unsqueeze(2).shape}')
        loss = nn.functional.mse_loss(output, x[:, 1:, 1].unsqueeze(2))  # Assuming y is the target

        loss.backward()
        optim.step()
        optim.zero_grad()
        r2 = 1 - (loss / torch.var(x[:, 1:, 1:2]))
        print(f'Epoch {epoch}, Loss: {loss.item()}, R2: {r2.item()}')

  0%|          | 0/157 [00:00<?, ?it/s]

Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 1.977914810180664, R2: -5.570251941680908
Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 1.5502876043319702, R2: -4.186545372009277
Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 1.17052161693573, R2: -2.879855155944824
Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 0.8730561137199402, R2: -1.9007277488708496
Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 0.6461938619613647, R2: -1.1230788230895996
Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 0.46868032217025757, R2: -0.5878821611404419
Output shape: torch.Size([64, 99, 1]), Target shape: torch.Size([64, 99, 1])
Epoch 0, Loss: 0.3642716109752655, R2: -0.22694003582000732
Output shape: torch.Size([64, 99, 1]), Target shape:

In [13]:
# validate results comparing overlay_trained to overlay
loss_benchmark = 0
loss_model = 0
var = 0
for x, y in tqdm(val_loader):
    x = x.float().to('cuda')
    output_benchmark = overlay(x)
    output_model = overlay_trained(x)
    loss_benchmark += ((output_benchmark - x[:, 1:, 1].unsqueeze(2)) ** 2).sum()
    loss_model += ((output_model - x[:, 1:, 1].unsqueeze(2)) ** 2).sum()
    var += torch.sum((x[:, 1:, 1:2] - x[:, 1:, 1:2].mean()) ** 2)
    print(f'Benchmark: {1 - loss_benchmark / var}, Model: {1 - loss_model / var}')


  0%|          | 0/63 [00:00<?, ?it/s]

Benchmark: 0.09975731372833252, Model: 0.1882210373878479
Benchmark: 0.06394553184509277, Model: 0.16972249746322632
Benchmark: 0.06455433368682861, Model: 0.1706632375717163
Benchmark: 0.07438045740127563, Model: 0.1778404712677002
Benchmark: 0.07547593116760254, Model: 0.18090271949768066
Benchmark: 0.07029324769973755, Model: 0.1813463568687439
Benchmark: 0.06840091943740845, Model: 0.18254268169403076
Benchmark: 0.06608045101165771, Model: 0.17925536632537842
Benchmark: 0.061960816383361816, Model: 0.1744307279586792
Benchmark: 0.06205528974533081, Model: 0.17557209730148315
Benchmark: 0.060156166553497314, Model: 0.1764112114906311
Benchmark: 0.062045156955718994, Model: 0.1785343885421753
Benchmark: 0.06327754259109497, Model: 0.17886847257614136
Benchmark: 0.06181693077087402, Model: 0.17851173877716064
Benchmark: 0.064067542552948, Model: 0.1784060001373291
Benchmark: 0.06127077341079712, Model: 0.17677205801010132
Benchmark: 0.06351757049560547, Model: 0.1781679391860962
Bench

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 5.79 GiB of which 38.00 MiB is free. Process 243534 has 532.00 MiB memory in use. Process 342514 has 122.00 MiB memory in use. Process 347077 has 154.00 MiB memory in use. Process 357453 has 552.00 MiB memory in use. Including non-PyTorch memory, this process has 3.65 GiB memory in use. Of the allocated memory 3.14 GiB is allocated by PyTorch, and 421.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)